# Tokenizers
> Imagine a word is an interval of time series data...   

In [ ]:
#| default_exp tokenizers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F, torch.nn as nn
from physiojepa.layers import Patch, InceptionBlock, MultiHeadAttention, get_activation_fn

In [ ]:
import torch.nn as nn
proj = nn.Conv1d(
            in_channels=3,
            out_channels=512*3, # if shared embedding, then d_model is the output dimension
            kernel_size=100,
            stride=100,
            padding=0,
            groups=3 # added to handle multiple channels / keep them separate
        )

import torch

W_P = nn.ModuleList()
for _ in range(3): W_P.append(nn.Linear(100, 512))

x = torch.randn(10,3,1000).shape

In [ ]:
l = nn.Linear(100,512)

In [ ]:
l.weight.shape, l.bias.shape

(torch.Size([512, 100]), torch.Size([512]))

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(W_P), count_parameters(proj)

(155136, 155136)

In [ ]:
#| export
class TS_Tokenizer(nn.Module):
    """
    Tokenizer class based on a Conv1D
    ---
        c_in (int): Number of input channels
        patch_size (int): Size of each patch/kernel
        d_model (int): Output embedding dimension
    """
    def __init__(self, c_in, patch_size, d_model, patch_stride=None, shared_embedding=True):
        super().__init__()
        self.c_in = c_in
        self.d_model = d_model
        self.patch_size = patch_size
        self.patch_stride = patch_stride if patch_stride is not None else patch_size
        self.shared_embedding = shared_embedding
        if not shared_embedding:
            assert d_model % c_in == 0, f"d_model ({d_model}) must be divisible by c_in ({c_in})"
        self.proj = nn.Conv1d(
            in_channels=c_in,
            out_channels=d_model, # if shared embedding, then d_model is the output dimension
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=0,
            groups=c_in if not shared_embedding else 1 # added to handle multiple channels / keep them separate
        )

    def forward(self, x):
        """
        Args:
            x: Either a regular tensor [batch_size, C, L] 
        Returns:
            Regular tensor [batch_size, num_patches, d_model]
        """
        bs, C, seq_len = x.shape
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=0.) # pad at the end with value
        x = self.proj(x)  # [batch_size, d_model * c_in, num_patches]
        if not self.shared_embedding:
            x = x.reshape(bs, self.c_in, self.d_model // self.c_in if not self.shared_embedding else self.d_model, -1)
            x = x.permute(0, 3, 1, 2) # [batch_size, num_patches, n_vars, d_model]
        else:
            x = x.transpose(1, 2) # [batch_size, num_patches, d_model]
        return x

class TS_Tokenizer_Complex(nn.Module):
    """
    Time series 2D convolutional Embedding
    """
    def __init__(
        self,
        c_in,
        patch_size,
        d_model,
        constant_pad_value=0.
    ):
        super().__init__()
        self.patch_size = patch_size
        self.constant_pad_value = constant_pad_value
        #self.simple_conv = nn.Conv1d(c_in, d_model, kernel_size=patch_size, stride=patch_size)
        self.emb_1 = nn.Conv2d(1, d_model*4, kernel_size=[1, patch_size], stride=[1, patch_size])
        self.emb_2 = nn.Conv2d(d_model*4, d_model, kernel_size=[c_in, 1])
    def forward(self, x):
        """
        Input: [bs x n channels x seq_len]
        Out:  [bs x num_patch x d_model]
        """
        bs, C, seq_len = x.shape
        #x = self.simple_conv(x).transpose(1, 2)
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=self.constant_pad_value) # pad at the end with value
        x = x.unsqueeze(1)
        x = self.emb_1(x) # [bs x d_model*4 x n channels x n patches]
        x = self.emb_2(x) # [bs x d_model x 1 x n patches]
        x = x.transpose(1, 3) 
        return x.squeeze(2)
    
class LinearTokenizer(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_size, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_embedding=False, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_embedding = shared_embedding
        self.n_vars = c_in
        self.patch_size = patch_size
        self.d_model = d_model
        self.patch = Patch(patch_len=self.patch_size, stride=self.patch_size)
        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_embedding:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(patch_size, d_model))
        else:
            self.W_P = nn.Linear(patch_size, d_model)
        #self.activation = nn.GELU()
        #self.project_out = nn.Linear(d_model*c_in, d_model)

    def forward(self, x):          
        """
        input: x: tensor [bs x nvars x seq_len]
        returns: x: tensor [bs x num_patch x d_model]
        """
        # Input embedding
        bs = x.shape[0]
        x = self.patch(x, constant_pad=True, constant_pad_value=0) # [bs x num_patch x n_vars x patch_len]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.n_vars):
                x_out.append(self.W_P[i](x[:,:,i,:]))
            x = torch.stack(x_out, dim=2)
        else:
            x = self.W_P(x) # x: [bs x num_patch x nvars x d_model]

        #x = self.activation(x)
        x = x.reshape(bs, -1, self.d_model)
        #x = x.flatten(start_dim=-2) # bs x n patch x nvars * d_model
        #x = self.project_out(x)
        return x 
    
class InceptionTokenizer(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_size, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 patch_stride=None, # the stride of the patches
                 shared_embedding=True,
                 **tokenizer_kwargs
                 ):
        super().__init__()
        self.n_vars = c_in
        self.patch_size = patch_size
        self.patch_stride = patch_stride if patch_stride is not None else patch_size
        self.d_model = d_model
        self.shared_embedding = shared_embedding
        self.inception = InceptionBlock(in_channels=c_in, groups=c_in if not shared_embedding else 1, **tokenizer_kwargs)
        self.inception_out = self.inception.bottleneck_channels * 4
        if not shared_embedding:
            assert d_model % c_in == 0, f"d_model ({d_model}) must be divisible by c_in ({c_in})"

        self.patch = nn.Conv1d(
            in_channels=self.inception_out,
            out_channels=d_model,
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=0,
            groups=self.n_vars if not shared_embedding else 1 # added to handle multiple channels / keep them separate
        )

    def forward(self, x):
        """
        input: x: tensor [bs x n channels x seq_len]
        returns: x: tensor [bs x num_patch x d_model]
        """
        bs, C, seq_len = x.shape
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=0.) # pad at the end with value
        x = self.inception(x) # [bs x d_model * n_vars (if not shared embedding) else d_model x seq_len]
        x = self.patch(x)
        if not self.shared_embedding:
            x = x.reshape(bs, self.n_vars, self.d_model // self.n_vars if not self.shared_embedding else self.d_model, -1)
            x = x.permute(0, 3, 1, 2)
        else:
            x = x.transpose(1, 2) # [bs x num_patch x num_vars xd_model]
        return x


In [ ]:
#| export
class MultiScaleTokenizer(nn.Module):
    """
    Hierarchical multi-scale tokenizer that combines fine-grained local encoding
    with coarse patch tokenization.
    
    Fine path: splits each coarse patch interval into `group_size` sub-patches,
    processes them with a lightweight local transformer, and pools each group
    into a single summary vector.
    
    Coarse path: standard TS_Tokenizer (Conv1d) over the full signal.
    
    Fusion: additive combination of projected fine summaries and coarse tokens.
    
    Output shape matches other tokenizers: [bs, num_patches, n_vars, d_model]
    ---
        c_in (int): Number of input channels
        patch_size (int): Coarse patch size (e.g. 125 for 1s at 125Hz)
        d_model (int): Output embedding dimension (coarse/global)
        patch_stride (int): Coarse patch stride (default=patch_size)
        shared_embedding (bool): Whether channels share embedding weights
        fine_patch_size (int): Fine sub-patch size in samples (default=25)
        fine_d_model (int): Fine encoder hidden dimension (default=128)
        fine_n_heads (int): Attention heads in local encoder (default=4)
        fine_d_ff (int): Feed-forward width in local encoder (default=512)
        fine_layers (int): Number of local transformer layers (default=1)
        fine_dropout (float): Dropout in local encoder (default=0.0)
    """
    def __init__(self,
                 c_in,
                 patch_size,
                 d_model,
                 patch_stride=None,
                 shared_embedding=True,
                 fine_patch_size=25,
                 fine_d_model=128,
                 fine_n_heads=4,
                 fine_d_ff=512,
                 fine_layers=1,
                 fine_dropout=0.0
                 ):
        super().__init__()
        self.c_in = c_in
        self.d_model = d_model
        self.patch_size = patch_size
        self.patch_stride = patch_stride if patch_stride is not None else patch_size
        self.shared_embedding = shared_embedding
        self.fine_patch_size = fine_patch_size
        self.fine_d_model = fine_d_model
        self.group_size = patch_size // fine_patch_size  # fine patches per coarse patch
        assert patch_size % fine_patch_size == 0, (
            f"patch_size ({patch_size}) must be divisible by fine_patch_size ({fine_patch_size})")
        
        # --- Coarse path: standard Conv1d tokenizer ---
        coarse_out_dim = d_model
        self.coarse_tokenizer = TS_Tokenizer(
            c_in=c_in,
            patch_size=patch_size,
            d_model=coarse_out_dim,
            patch_stride=self.patch_stride,
            shared_embedding=shared_embedding
        )
        
        # --- Fine path ---
        # Fine tokenizer: channel-specific grouped Conv1d at fine resolution
        fine_conv_out = fine_d_model * c_in if not shared_embedding else fine_d_model
        self.fine_proj = nn.Conv1d(
            in_channels=c_in,
            out_channels=fine_conv_out,
            kernel_size=fine_patch_size,
            stride=fine_patch_size,
            padding=0,
            groups=c_in if not shared_embedding else 1
        )
        
        # Local transformer: processes groups of fine patches
        self.local_layers = nn.ModuleList([
            _LocalTSTBlock(
                d_model=fine_d_model,
                n_heads=fine_n_heads,
                d_ff=fine_d_ff,
                dropout=fine_dropout
            ) for _ in range(fine_layers)
        ])
        
        # Attention pooling over fine patches within each group
        self.pool_query = nn.Parameter(torch.randn(1, 1, fine_d_model) * 0.02)
        self.pool_attn = nn.Linear(fine_d_model, 1)
        
        # Fusion: project fine summaries into coarse token space
        per_channel_d = d_model // c_in if not shared_embedding else d_model
        self.fine_to_coarse = nn.Linear(fine_d_model, per_channel_d)
        self.fusion_norm = nn.LayerNorm(per_channel_d)
        self.fusion_gate = nn.Parameter(torch.tensor(0.5))
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, C, L] raw waveform
        Returns:
            [batch_size, num_patches, n_vars, d_model//n_vars] if not shared_embedding
            [batch_size, num_patches, 1, d_model] if shared_embedding
        """
        bs, C, seq_len = x.shape
        
        # --- Coarse path ---
        coarse = self.coarse_tokenizer(x)  # [bs, num_patches, n_vars, d_per_ch] or [bs, num_patches, d_model]
        
        # --- Fine path ---
        # Pad if needed (same logic as coarse)
        if ((seq_len - self.patch_size) % self.patch_size != 0):
            x_fine = F.pad(x, (0, self.patch_size), 'constant', value=0.)
        else:
            x_fine = x
        
        fine = self.fine_proj(x_fine)  # [bs, fine_conv_out, total_fine_patches]
        
        if not self.shared_embedding:
            # Reshape to [bs, C, fine_d_model//... wait, grouped conv output is fine_d_model*C
            # fine_proj output: [bs, fine_d_model * C, total_fine_patches]
            total_fine_patches = fine.shape[-1]
            num_patches = total_fine_patches // self.group_size
            # Reshape: [bs, C, fine_d_model, total_fine_patches]
            fine = fine.reshape(bs, C, self.fine_d_model, total_fine_patches)
            # -> [bs, C, total_fine_patches, fine_d_model]
            fine = fine.permute(0, 1, 3, 2)
            # -> [bs, C, num_patches, group_size, fine_d_model]
            fine = fine.reshape(bs, C, num_patches, self.group_size, self.fine_d_model)
            # Process each channel independently through local transformer
            # Merge bs and C and num_patches: [bs*C*num_patches, group_size, fine_d_model]
            fine = fine.reshape(bs * C * num_patches, self.group_size, self.fine_d_model)
        else:
            # fine_proj output: [bs, fine_d_model, total_fine_patches]
            total_fine_patches = fine.shape[-1]
            num_patches = total_fine_patches // self.group_size
            fine = fine.transpose(1, 2)  # [bs, total_fine_patches, fine_d_model]
            fine = fine.reshape(bs, num_patches, self.group_size, self.fine_d_model)
            fine = fine.reshape(bs * num_patches, self.group_size, self.fine_d_model)
        
        # Local transformer over group_size tokens
        for layer in self.local_layers:
            fine = layer(fine)  # [N, group_size, fine_d_model]
        
        # Attention pooling: weighted sum over group_size dimension
        attn_logits = self.pool_attn(fine).squeeze(-1)  # [N, group_size]
        attn_weights = F.softmax(attn_logits, dim=-1).unsqueeze(-1)  # [N, group_size, 1]
        fine_pooled = (fine * attn_weights).sum(dim=1)  # [N, fine_d_model]
        
        # Project to coarse dimension
        fine_projected = self.fine_to_coarse(fine_pooled)  # [N, per_channel_d]
        fine_projected = self.fusion_norm(fine_projected)
        
        # --- Fusion ---
        gate = torch.sigmoid(self.fusion_gate)
        
        if not self.shared_embedding:
            # fine_projected: [bs*C*num_patches, d_model//C]
            per_ch_d = self.d_model // self.c_in
            fine_projected = fine_projected.reshape(bs, C, num_patches, per_ch_d)
            fine_projected = fine_projected.permute(0, 2, 1, 3)  # [bs, num_patches, C, per_ch_d]
            # coarse is [bs, num_patches, C, per_ch_d]
            fused = gate * coarse + (1 - gate) * fine_projected
        else:
            # fine_projected: [bs*num_patches, d_model]
            fine_projected = fine_projected.reshape(bs, num_patches, self.d_model)
            # coarse is [bs, num_patches, d_model]
            fused = gate * coarse + (1 - gate) * fine_projected
        
        return fused


class _LocalTSTBlock(nn.Module):
    """Lightweight transformer block for local (intra-group) attention."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.attn = MultiHeadAttention(
            dim=d_model, num_heads=n_heads,
            qkv_bias=True, attn_drop=dropout, proj_drop=dropout,
            rotary_pes=False  # groups are small, no positional encoding needed
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Pre-norm style
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

In [ ]:
#| export
class PatchEncoder(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_len, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_embedding, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_embedding = shared_embedding
        self.n_vars = c_in
        self.patch_len = patch_len
        self.d_model = d_model

        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_embedding:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(patch_len, d_model))
        else:
            self.W_P = nn.Linear(patch_len, d_model)

    def forward(self, x):          
        """
        input: x: tensor [bs x num_patch x nvars x patch_len]
        returns: x: tensor [bs x num_patch x nvars x d_model]
        """
        # Input embedding
        if not self.shared_embedding:
            x_out = []
            for i in range(self.n_vars):
                x_out.append(self.W_P[i](x[:,:,i,:]))
            x = torch.stack(x_out, dim=2)
        else:
            x = self.W_P(x) # x: [bs x num_patch x nvars x d_model]
        return x

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()